# State of Data Brasil 2025 — Gold
### Tech Challenge Fase 3 — Grupo 6
### Edição 2025

Fonte: [Kaggle — State of Data Brasil](https://www.kaggle.com/datahackers/datasets)

A Gold é a camada de **consumo**: responde as 7 perguntas de negócio do desafio a
partir da Silver e persiste cada resposta como uma tabela pronta pra gráfico. Cada
tabela sai particionada por `ano_pesquisa`, no mesmo caminho compartilhado — assim
o material executivo lê a Gold das edições juntas e compara ano a ano.

## 1. Preparando para as consultas SQL

In [1]:
import sys
from pathlib import Path

CAMINHO_ATUAL = Path.cwd().resolve()
for caminho in [CAMINHO_ATUAL, *CAMINHO_ATUAL.parents]:
    if (caminho / "utils" / "config.py").exists():
        RAIZ_PROJETO = caminho
        break
else:
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto (utils/config.py).")
if str(RAIZ_PROJETO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROJETO))

from utils.config import CAMINHO_SILVER, CAMINHO_GOLD_BASE
print("Silver:", CAMINHO_SILVER)
print("Gold:", CAMINHO_GOLD_BASE)

Silver: C:\Users\Henrique\Desktop\Tech Challenge 3 local\data\silver\state_of_data_silver
Gold: C:\Users\Henrique\Desktop\Tech Challenge 3 local\data\gold


In [2]:
from pyspark.sql import SparkSession, functions as F
from utils.constants import ponto_medio_salarial, ordem_senioridade, ordem_tempo_experiencia

ANO_PESQUISA = 2025
spark = SparkSession.builder.appName(f"state-of-data-{ANO_PESQUISA}-gold").getOrCreate()
print(f"Ano de pesquisa: {ANO_PESQUISA}")

c:\Users\Henrique\AppData\Local\Programs\Python\Python314\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Ano de pesquisa: 2025


Leio a Silver e filtro só a minha edição (o caminho é compartilhado, particionado
por `ano_pesquisa`). Registro como view temporária pra consultar em SQL.

In [3]:
df_silver = (
    spark.read.parquet(str(CAMINHO_SILVER))
    .filter(F.col("ano_pesquisa") == ANO_PESQUISA)
)
print(f"{df_silver.count()} linhas, {len(df_silver.columns)} colunas")
df_silver.createOrReplaceTempView("state_of_data")

3494 linhas, 170 colunas


## 2. Regras de referência para as consultas

**Filtros de escopo** (colunas booleanas geradas na Silver — evitam repetir a
lista de "não se aplica" em toda query):

| Flag | Usar em |
| :-- | :-- |
| `aplica_analise_emprego` | setor, salário, experiência, modelo de trabalho, satisfação |
| `aplica_analise_tecnica` | cargo, senioridade e os blocos técnicos (linguagens, banco, cloud, BI, IA técnica) |
| `aplica_analise_gestor` | cargos no time, desafios e responsabilidades de gestor |
| `satisfeito_empresa = false` | motivo de insatisfação |

**Contagem de múltipla escolha:** as colunas de bloco são boolean. Pra contar quem
marcou uma opção uso `SUM(CASE WHEN coluna THEN 1 ELSE 0 END)`, nunca `COUNT`
(que contaria também os `false`).

**Viés de composição:** ao comparar médias salariais entre grupos (região, gênero),
sempre abro também por senioridade — um grupo com mais gente sênior sobe a média
por composição, não por um efeito real do grupo.

A faixa salarial é uma categoria de texto. Pra calcular médias e comparar grupos,
crio uma view auxiliar com a faixa convertida para o ponto médio em R$
(`faixa_salarial_num`), usando o de-para de `utils/constants.py`.

In [4]:
case_faixa_salarial = " ".join(
    f"WHEN '{faixa}' THEN {valor}" for faixa, valor in ponto_medio_salarial.items()
)
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW state_of_data_num AS
    SELECT *,
        CASE faixa_salarial {case_faixa_salarial} ELSE NULL END AS faixa_salarial_num
    FROM state_of_data
""")
print("Views criadas: state_of_data e state_of_data_num")

Views criadas: state_of_data e state_of_data_num


Função de gráfico de barras horizontais reutilizada em todas as perguntas (recebe
um pandas vindo de `toPandas()`). Uso backend `Agg` pra o notebook rodar mesmo sem
display (ex: execução automatizada).

In [5]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import warnings
warnings.filterwarnings("ignore")
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")

def grafico_barh(pdf, col_categoria, col_valor, titulo, xlabel, cor="#2E86AB", figsize=(9, 5)):
    pdf = pdf.sort_values(col_valor)
    plt.figure(figsize=figsize)
    plt.barh(pdf[col_categoria].astype(str), pdf[col_valor], color=cor)
    plt.title(titulo)
    plt.xlabel(xlabel)
    plt.tight_layout()
    plt.show()

def soma_multipla_escolha(rotulos_colunas, escopo="aplica_analise_tecnica"):
    """Monta uma query UNION ALL somando cada opção de um bloco de múltipla
    escolha (boolean) numa linha (rotulo, total). Evita repetir SQL na mão."""
    partes = [
        f"SELECT '{rotulo}' AS categoria, "
        f"SUM(CASE WHEN {coluna} THEN 1 ELSE 0 END) AS total "
        f"FROM state_of_data WHERE {escopo} = true"
        for rotulo, coluna in rotulos_colunas
    ]
    return " UNION ALL ".join(partes) + " ORDER BY total DESC"

## 3. Respondendo as perguntas do Tech Challenge

### P1 — Como está estruturado o mercado brasileiro de Dados?

**1. Distribuição da situação de trabalho.** Pergunta de perfil, respondida por
todos — sem filtro de escopo.

In [6]:
resultado_p1_1 = spark.sql("""
    SELECT situacao_trabalho, COUNT(*) AS total,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS percentual
    FROM state_of_data
    GROUP BY situacao_trabalho
    ORDER BY total DESC
""")
resultado_p1_1.show(truncate=False)
grafico_barh(resultado_p1_1.toPandas(), "situacao_trabalho", "total",
             "Situação de trabalho dos respondentes 2025", "Quantidade de pessoas")

+---------------------------------------------------------------+-----+----------+
|situacao_trabalho                                              |total|percentual|
+---------------------------------------------------------------+-----+----------+
|Empregado (CLT)                                                |2278 |65.2      |
|Empreendedor ou Empregado (CNPJ)                               |428  |12.2      |
|Desempregado, buscando recolocação                             |167  |4.8       |
|Vivo no Brasil e trabalho remoto para empresa de fora do Brasil|141  |4.0       |
|Estagiário                                                     |126  |3.6       |
|Servidor Público                                               |123  |3.5       |
|Vivo fora do Brasil e trabalho para empresa de fora do Brasil  |66   |1.9       |
|Trabalho na área Acadêmica/Pesquisador                         |44   |1.3       |
|Freelancer                                                     |37   |1.1       |
|Som

**2. Top 10 setores de atuação.** Setor é pergunta sobre a empresa atual, então
filtro por `aplica_analise_emprego` (exclui quem não tem vínculo).

In [7]:
resultado_p1_2 = spark.sql("""
    SELECT setor_empresa, COUNT(*) AS total
    FROM state_of_data
    WHERE aplica_analise_emprego = true
    GROUP BY setor_empresa ORDER BY total DESC LIMIT 10
""")
resultado_p1_2.show(truncate=False)
grafico_barh(resultado_p1_2.toPandas(), "setor_empresa", "total",
             "Top 10 setores de atuação 2025", "Quantidade de pessoas")

+------------------------------+-----+
|setor_empresa                 |total|
+------------------------------+-----+
|Finanças ou Bancos            |598  |
|Tecnologia/Fábrica de Software|563  |
|Área de Consultoria           |285  |
|Outra Opção                   |243  |
|Indústria                     |237  |
|Varejo                        |181  |
|Educação                      |163  |
|Setor Público                 |136  |
|Área da Saúde                 |128  |
|Internet/Ecommerce            |105  |
+------------------------------+-----+



**3. Distribuição de cargos atuais.** Cargo só existe pra quem atua tecnicamente,
então filtro por `aplica_analise_tecnica`.

In [8]:
resultado_p1_3 = spark.sql("""
    SELECT cargo_atual, COUNT(*) AS total
    FROM state_of_data WHERE aplica_analise_tecnica = true
    GROUP BY cargo_atual ORDER BY total DESC
""")
resultado_p1_3.show(truncate=False)
grafico_barh(resultado_p1_3.toPandas(), "cargo_atual", "total",
             "Distribuição de cargos 2025", "Total")

+-----------------------------------------------------------+-----+
|cargo_atual                                                |total|
+-----------------------------------------------------------+-----+
|Analista de Dados/Data Analyst                             |599  |
|Cientista de Dados/Data Scientist                          |423  |
|Engenheiro de Dados/Data Engineer/Data Architect           |402  |
|Analista de BI/BI Analyst                                  |215  |
|Outra Opção                                                |206  |
|Analista de Negócios/Business Analyst                      |141  |
|Analytics Engineer                                         |135  |
|Desenvolvedor/ Engenheiro de Software/ Analista de Sistemas|106  |
|Engenheiro de Machine Learning/ML Engineer/AI Engineer     |106  |
|Analista de Suporte/Analista Técnico                       |55   |
|Data Product Manager/ Product Manager (PM/APM/DPM/GPM/PO)  |33   |
|Arquiteto de Dados/Data Architect              

**4. Distribuição de senioridade.** Mesmo escopo técnico. Nesta edição existe o
nível "Especialista/Staff+" além de Júnior/Pleno/Sênior — a query não filtra
níveis, então ele aparece no resultado.

In [9]:
resultado_p1_4 = spark.sql("""
    SELECT nivel_senioridade, COUNT(*) AS total,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 0) AS percentual
    FROM state_of_data WHERE aplica_analise_tecnica = true
    GROUP BY nivel_senioridade ORDER BY total DESC
""")
resultado_p1_4.show(truncate=False)
grafico_barh(resultado_p1_4.toPandas(), "nivel_senioridade", "total",
             "Distribuição de senioridade 2025", "Total")

+-------------------+-----+----------+
|nivel_senioridade  |total|percentual|
+-------------------+-----+----------+
|Sênior             |858  |34        |
|Pleno              |775  |31        |
|Júnior             |518  |21        |
|Especialista/Staff+|349  |14        |
+-------------------+-----+----------+



**5. % de profissionais que atuam como gestor.** Sobre a base empregada
(`aplica_analise_emprego`), qual fração é gestora. Resultado vira um card.

In [10]:
resultado_p1_5 = spark.sql("""
    SELECT ROUND(100.0 * SUM(CASE WHEN aplica_analise_gestor THEN 1 ELSE 0 END) / COUNT(*), 0) AS pct_gestores
    FROM state_of_data WHERE aplica_analise_emprego = true
""")
resultado_p1_5.show(truncate=False)
pct_val = resultado_p1_5.collect()[0]["pct_gestores"]
fig, ax = plt.subplots(figsize=(4, 2.2)); ax.axis("off")
ax.add_patch(patches.FancyBboxPatch((0.05, 0.05), 0.9, 0.9, boxstyle="round,pad=0.03",
             facecolor="#F8F9FA", edgecolor="#D0D7DE", linewidth=1.5))
ax.text(0.5, 0.58, f"{int(pct_val)}%", fontsize=38, fontweight="bold", ha="center", va="center", color="#0969DA")
ax.text(0.5, 0.28, "% dos profissionais que atuam como gestores 2025", fontsize=9, ha="center", va="center", color="#57606A")
plt.tight_layout(); plt.show()

+------------+
|pct_gestores|
+------------+
|23          |
+------------+



**6. Relação tempo de experiência em dados x senioridade.** Cruza as duas colunas;
escopo técnico + empregado.

In [11]:
resultado_p1_6 = spark.sql("""
    SELECT tempo_experiencia_dados, nivel_senioridade, COUNT(*) AS total
    FROM state_of_data
    WHERE aplica_analise_emprego = true AND aplica_analise_tecnica = true
    GROUP BY tempo_experiencia_dados, nivel_senioridade
    ORDER BY tempo_experiencia_dados, total DESC
""")
resultado_p1_6.show(50, truncate=False)

+--------------------------------------+-------------------+-----+
|tempo_experiencia_dados               |nivel_senioridade  |total|
+--------------------------------------+-------------------+-----+
|Mais de 10 anos                       |Especialista/Staff+|99   |
|Mais de 10 anos                       |Sênior             |78   |
|Mais de 10 anos                       |Pleno              |13   |
|Mais de 10 anos                       |Júnior             |1    |
|Menos de 1 ano                        |Júnior             |147  |
|Menos de 1 ano                        |Pleno              |34   |
|Menos de 1 ano                        |Sênior             |13   |
|Menos de 1 ano                        |Especialista/Staff+|1    |
|Não tenho experiência na área de dados|Júnior             |68   |
|Não tenho experiência na área de dados|Pleno              |33   |
|Não tenho experiência na área de dados|Sênior             |31   |
|Não tenho experiência na área de dados|Especialista/Staff+|4 

**7. Modelo de trabalho atual e o ideal.** Duas tabelas (atual e desejado),
ambas sobre a base empregada.

In [12]:
import textwrap
resultado_p1_7 = spark.sql("""
    SELECT modelo_trabalho_atual, COUNT(*) AS total
    FROM state_of_data WHERE aplica_analise_emprego = true
    GROUP BY modelo_trabalho_atual ORDER BY total DESC
""")
resultado_p1_7.show(truncate=False)
pdf = resultado_p1_7.toPandas()
pdf["fmt"] = pdf["modelo_trabalho_atual"].apply(lambda x: textwrap.fill(str(x), 30))
grafico_barh(pdf, "fmt", "total", "Modelo de trabalho atual 2025", "Quantidade de pessoas")

+--------------------------------------------------------------------------------------------------------------+-----+
|modelo_trabalho_atual                                                                                         |total|
+--------------------------------------------------------------------------------------------------------------+-----+
|Modelo 100% remoto                                                                                            |1281 |
|Modelo 100% presencial                                                                                        |670  |
|Modelo híbrido com dias fixos de trabalho presencial                                                          |645  |
|Modelo híbrido flexível (o funcionário tem liberdade para escolher quando estar no escritório presencialmente)|631  |
+--------------------------------------------------------------------------------------------------------------+-----+



In [13]:
resultado_p1_7_1 = spark.sql("""
    SELECT modelo_trabalho_ideal, COUNT(*) AS total
    FROM state_of_data WHERE aplica_analise_emprego = true
    GROUP BY modelo_trabalho_ideal ORDER BY total DESC
""")
resultado_p1_7_1.show(truncate=False)
pdf = resultado_p1_7_1.toPandas()
pdf["fmt"] = pdf["modelo_trabalho_ideal"].apply(lambda x: textwrap.fill(str(x), 30))
grafico_barh(pdf, "fmt", "total", "Modelo de trabalho ideal 2025", "Quantidade de pessoas")

+--------------------------------------------------------------------------------------------------------------+-----+
|modelo_trabalho_ideal                                                                                         |total|
+--------------------------------------------------------------------------------------------------------------+-----+
|Modelo híbrido flexível (o funcionário tem liberdade para escolher quando estar no escritório presencialmente)|1387 |
|Modelo 100% remoto                                                                                            |1359 |
|Modelo híbrido com dias fixos de trabalho presencial                                                          |421  |
|Modelo 100% presencial                                                                                        |60   |
+--------------------------------------------------------------------------------------------------------------+-----+



**8. Porte das empresas (nº de funcionários).** Sobre a base empregada; ignoro
nulos (inclui a categoria ambígua que a Silver zerou).

In [14]:
resultado_p1_8 = spark.sql("""
    SELECT num_funcionarios, COUNT(*) AS total
    FROM state_of_data
    WHERE aplica_analise_emprego = true AND num_funcionarios IS NOT NULL
    GROUP BY num_funcionarios ORDER BY total DESC
""")
resultado_p1_8.show(20, truncate=False)
grafico_barh(resultado_p1_8.toPandas(), "num_funcionarios", "total",
             "Porte da empresa 2025", "Quantidade de pessoas")

+----------------+-----+
|num_funcionarios|total|
+----------------+-----+
|Acima de 3.000  |1318 |
|de 101 a 500    |538  |
|de 1.001 a 3.000|378  |
|de 501 a 1.000  |342  |
|de 51 a 100     |270  |
|de 11 a 50      |234  |
|de 1 a 5        |95   |
|de 6 a 10       |51   |
+----------------+-----+



### P2 — Quais perfis são mais valorizados?

**1. Faixa salarial por cargo e senioridade** (escopo empregado + técnico).

In [15]:
resultado_p2_1 = spark.sql("""
    SELECT cargo_atual, nivel_senioridade, faixa_salarial, COUNT(*) AS total
    FROM state_of_data
    WHERE aplica_analise_emprego = true AND aplica_analise_tecnica = true
    GROUP BY cargo_atual, nivel_senioridade, faixa_salarial
    ORDER BY cargo_atual, nivel_senioridade, total DESC
""")
resultado_p2_1.show(100, truncate=False)

+-------------------------------------+-------------------+--------------------------------+-----+
|cargo_atual                          |nivel_senioridade  |faixa_salarial                  |total|
+-------------------------------------+-------------------+--------------------------------+-----+
|Analista de BI/BI Analyst            |Especialista/Staff+|de R$ 8.001/mês a R$ 12.000/mês |5    |
|Analista de BI/BI Analyst            |Especialista/Staff+|de R$ 12.001/mês a R$ 16.000/mês|5    |
|Analista de BI/BI Analyst            |Especialista/Staff+|de R$ 30.001/mês a R$ 40.000/mês|2    |
|Analista de BI/BI Analyst            |Especialista/Staff+|de R$ 6.001/mês a R$ 8.000/mês  |2    |
|Analista de BI/BI Analyst            |Especialista/Staff+|de R$ 16.001/mês a R$ 20.000/mês|1    |
|Analista de BI/BI Analyst            |Especialista/Staff+|de R$ 25.001/mês a R$ 30.000/mês|1    |
|Analista de BI/BI Analyst            |Especialista/Staff+|de R$ 20.001/mês a R$ 25.000/mês|1    |
|Analista 

**2. Salário médio estimado por tempo de experiência (Dados e TI).** Uso a faixa
convertida em R$ e ordeno pelas faixas de tempo (de `utils/constants.py`).

In [16]:
import textwrap
case_ordem_exp = " ".join(f"WHEN '{v}' THEN {i}" for i, v in enumerate(ordem_tempo_experiencia))

resultado_p2_2 = spark.sql(f"""
    SELECT tempo_experiencia_dados, ROUND(AVG(faixa_salarial_num), 0) AS salario_medio_estimado
    FROM state_of_data_num WHERE aplica_analise_emprego = true
    GROUP BY tempo_experiencia_dados
    ORDER BY CASE tempo_experiencia_dados {case_ordem_exp} ELSE 99 END ASC
""")
resultado_p2_2.show(truncate=False)
pdf = resultado_p2_2.toPandas()
pdf["fmt"] = pdf["tempo_experiencia_dados"].apply(lambda x: textwrap.fill(str(x), 30))
grafico_barh(pdf, "fmt", "salario_medio_estimado",
             "Salário médio por experiência em dados 2025", "Salário médio estimado (R$)")

resultado_p2_2_1 = spark.sql(f"""
    SELECT tempo_experiencia_ti, ROUND(AVG(faixa_salarial_num), 0) AS salario_medio_estimado
    FROM state_of_data_num WHERE aplica_analise_emprego = true
    GROUP BY tempo_experiencia_ti
    ORDER BY CASE tempo_experiencia_ti {case_ordem_exp} ELSE 99 END ASC
""")
resultado_p2_2_1.show(truncate=False)
pdf = resultado_p2_2_1.toPandas()
pdf["fmt"] = pdf["tempo_experiencia_ti"].apply(lambda x: textwrap.fill(str(x), 30))
grafico_barh(pdf, "fmt", "salario_medio_estimado",
             "Salário médio por experiência em TI 2025", "Salário médio estimado (R$)")

+--------------------------------------+----------------------+
|tempo_experiencia_dados               |salario_medio_estimado|
+--------------------------------------+----------------------+
|Não tenho experiência na área de dados|7875.0                |
|Menos de 1 ano                        |4708.0                |
|de 1 a 2 anos                         |6997.0                |
|de 3 a 4 anos                         |10702.0               |
|de 5 a 6 anos                         |15108.0               |
|de 7 a 10 anos                        |19939.0               |
|Mais de 10 anos                       |22376.0               |
+--------------------------------------+----------------------+

+-------------------------------------------------------------------------------------------------------+----------------------+
|tempo_experiencia_ti                                                                                   |salario_medio_estimado|
+------------------------------------

**3. Migrou de TI x começou direto em dados.** Uso o texto exato de "não tive
experiência em TI" pra separar os dois grupos e comparar o salário médio.

In [17]:
resultado_p2_3 = spark.sql("""
    SELECT
        CASE WHEN tempo_experiencia_ti =
            'Não tive experiência na área de TI/Engenharia de Software antes de começar a trabalhar na área de dados'
            THEN 'Começou direto em dados' ELSE 'Migrou de TI' END AS origem,
        ROUND(AVG(faixa_salarial_num), 0) AS salario_medio_estimado, COUNT(*) AS total
    FROM state_of_data_num
    WHERE aplica_analise_emprego = true AND tempo_experiencia_ti IS NOT NULL
    GROUP BY 1 ORDER BY total DESC
""")
resultado_p2_3.show(truncate=False)
grafico_barh(resultado_p2_3.toPandas(), "origem", "salario_medio_estimado",
             "Salário médio por origem na área 2025", "Salário médio estimado (R$)")

+-----------------------+----------------------+-----+
|origem                 |salario_medio_estimado|total|
+-----------------------+----------------------+-----+
|Começou direto em dados|11888.0               |1745 |
|Migrou de TI           |14804.0               |1482 |
+-----------------------+----------------------+-----+



**4. Salário por função (Eng./Analista/Cientista) e senioridade.** Filtro os três
cargos principais e ordeno a senioridade logicamente pro gráfico.

In [18]:
import textwrap, pandas as pd
resultado_p2_4 = spark.sql("""
    SELECT cargo_atual, nivel_senioridade, ROUND(AVG(faixa_salarial_num), 0) AS salario_medio_estimado
    FROM state_of_data_num
    WHERE aplica_analise_emprego = true AND aplica_analise_tecnica = true
      AND cargo_atual IN (
          'Engenheiro de Dados/Data Engineer/Data Architect',
          'Analista de Dados/Data Analyst',
          'Cientista de Dados/Data Scientist')
    GROUP BY cargo_atual, nivel_senioridade
""")
resultado_p2_4.show(truncate=False)
pdf = resultado_p2_4.toPandas()
mapa = {'Analista de Dados/Data Analyst': 'Analista de Dados',
        'Cientista de Dados/Data Scientist': 'Cientista de Dados',
        'Engenheiro de Dados/Data Engineer/Data Architect': 'Engenheiro de Dados'}
pdf["cargo_curto"] = pdf["cargo_atual"].map(mapa)
pdf["nivel_senioridade"] = pd.Categorical(pdf["nivel_senioridade"], categories=ordem_senioridade, ordered=True)
pdf = pdf.sort_values(["cargo_curto", "nivel_senioridade"])
pdf["cargo_senioridade"] = pdf["cargo_curto"] + " - " + pdf["nivel_senioridade"].astype(str)
pdf["fmt"] = pdf["cargo_senioridade"].apply(lambda x: textwrap.fill(str(x), 30))
grafico_barh(pdf, "fmt", "salario_medio_estimado",
             "Salário médio por cargo e senioridade 2025", "Salário médio estimado (R$)")

+------------------------------------------------+-------------------+----------------------+
|cargo_atual                                     |nivel_senioridade  |salario_medio_estimado|
+------------------------------------------------+-------------------+----------------------+
|Engenheiro de Dados/Data Engineer/Data Architect|Sênior             |16986.0               |
|Analista de Dados/Data Analyst                  |Pleno              |7436.0                |
|Cientista de Dados/Data Scientist               |Pleno              |9644.0                |
|Analista de Dados/Data Analyst                  |Júnior             |3926.0                |
|Cientista de Dados/Data Scientist               |Sênior             |15724.0               |
|Engenheiro de Dados/Data Engineer/Data Architect|Júnior             |4526.0                |
|Analista de Dados/Data Analyst                  |Sênior             |12179.0               |
|Engenheiro de Dados/Data Engineer/Data Architect|Pleno     

**5. Objetivos de carreira mais citados (top 5).** Esta pergunta é respondida
sobretudo por quem está fora do mercado ativo, então uso
`aplica_analise_emprego = false`.

In [19]:
resultado_p2_5 = spark.sql("""
    SELECT objetivo_carreira, COUNT(*) AS total
    FROM state_of_data
    WHERE aplica_analise_emprego = false AND objetivo_carreira IS NOT NULL
    GROUP BY objetivo_carreira ORDER BY total DESC LIMIT 5
""")
resultado_p2_5.show(truncate=False)

+-------------------------------------------------------------------------------------------------------------------------+-----+
|objetivo_carreira                                                                                                        |total|
+-------------------------------------------------------------------------------------------------------------------------+-----+
|Preparação profissional: Estou buscando conhecimentos técnicos para no futuro assumir algum cargo na área de dados       |100  |
|Migração de carreira: Trabalho em outra área e busco recolocação na área de dados                                        |75   |
|Primeiro emprego: Estou tentando encontrar a primeira oportunidade na área de dados                                      |42   |
|Apenas conhecimentos: Busco conhecimentos em dados para utilizar na minha área de atuação (não trabalho na área de dados)|19   |
|Trabalho na área                                                                         

### P3 — Diversidade de gênero

**1. Proporção de gênero** (perfil, sem escopo).

In [20]:
resultado_p3_1 = spark.sql("""
    SELECT genero, COUNT(*) AS total,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS percentual
    FROM state_of_data GROUP BY genero ORDER BY total DESC
""")
resultado_p3_1.show(truncate=False)
grafico_barh(resultado_p3_1.toPandas(), "genero", "total", "Distribuição de gênero 2025", "Total")

+--------------------+-----+----------+
|genero              |total|percentual|
+--------------------+-----+----------+
|Masculino           |2707 |77.5      |
|Feminino            |767  |22.0      |
|Prefiro não informar|13   |0.4       |
|Outro               |7    |0.2       |
+--------------------+-----+----------+



**2. Gap salarial por gênero no mesmo cargo e senioridade** (controla cargo +
senioridade pra isolar o efeito do gênero).

In [21]:
resultado_p3_2 = spark.sql("""
    SELECT cargo_atual, nivel_senioridade, genero, ROUND(AVG(faixa_salarial_num), 0) AS salario_medio_estimado
    FROM state_of_data_num
    WHERE aplica_analise_emprego = true AND aplica_analise_tecnica = true
    GROUP BY cargo_atual, nivel_senioridade, genero
    ORDER BY cargo_atual, nivel_senioridade, genero
""")
resultado_p3_2.show(100, truncate=False)

+-----------------------------------------------------------+-------------------+--------------------+----------------------+
|cargo_atual                                                |nivel_senioridade  |genero              |salario_medio_estimado|
+-----------------------------------------------------------+-------------------+--------------------+----------------------+
|Analista de BI/BI Analyst                                  |Especialista/Staff+|Feminino            |18250.0               |
|Analista de BI/BI Analyst                                  |Especialista/Staff+|Masculino           |14773.0               |
|Analista de BI/BI Analyst                                  |Júnior             |Feminino            |3333.0                |
|Analista de BI/BI Analyst                                  |Júnior             |Masculino           |4106.0                |
|Analista de BI/BI Analyst                                  |Júnior             |Outro               |1500.0          

**3. Cor/raça/etnia por nível de senioridade** (escopo técnico).

In [22]:
resultado_p3_3 = spark.sql("""
    SELECT nivel_senioridade, cor_raca_etnia, COUNT(*) AS total
    FROM state_of_data WHERE aplica_analise_tecnica = true
    GROUP BY nivel_senioridade, cor_raca_etnia ORDER BY nivel_senioridade, total DESC
""")
resultado_p3_3.show(50, truncate=False)

+-------------------+--------------------+-----+
|nivel_senioridade  |cor_raca_etnia      |total|
+-------------------+--------------------+-----+
|Especialista/Staff+|Branca              |244  |
|Especialista/Staff+|Parda               |67   |
|Especialista/Staff+|Preta               |23   |
|Especialista/Staff+|Amarela             |10   |
|Especialista/Staff+|Prefiro não informar|3    |
|Especialista/Staff+|NULL                |1    |
|Especialista/Staff+|Outra               |1    |
|Júnior             |Branca              |303  |
|Júnior             |Parda               |148  |
|Júnior             |Preta               |42   |
|Júnior             |Amarela             |19   |
|Júnior             |Prefiro não informar|4    |
|Júnior             |Outra               |2    |
|Pleno              |Branca              |490  |
|Pleno              |Parda               |198  |
|Pleno              |Preta               |53   |
|Pleno              |Amarela             |28   |
|Pleno              

### P4 — Tecnologias mais adotadas

**1. Linguagens mais usadas.** Bloco de múltipla escolha (escopo técnico): somo cada
opção com `SUM(CASE WHEN ... THEN 1 ELSE 0 END)`. Monto a query a partir da lista de
colunas do bloco pra não repetir SQL na mão.

In [29]:
linguagens = [
    ("SQL", "linguagens_trabalho__sql"),
    ("Python", "linguagens_trabalho__python"),
    ("R", "linguagens_trabalho__r"),
    ("Java", "linguagens_trabalho__java"),
    ("JavaScript", "linguagens_trabalho__javascript"),
    ("C/C++/C#", "linguagens_trabalho__c_c_c"),
    (".NET", "linguagens_trabalho__net"),
    ("Scala", "linguagens_trabalho__scala"),
    ("VBA", "linguagens_trabalho__visual_basic_vba"),
    ("SAS/Stata", "linguagens_trabalho__sas_stata"),
    ("PHP", "linguagens_trabalho__php"),
    ("Julia", "linguagens_trabalho__julia"),
    ("Matlab", "linguagens_trabalho__matlab"),
    ("Rust", "linguagens_trabalho__rust"),
    ("Não utiliza", "linguagens_trabalho__nao_utilizo_nenhuma_linguagem"),
]
resultado_p4_1 = spark.sql(soma_multipla_escolha(linguagens)).withColumnRenamed("categoria", "linguagem")
resultado_p4_1.show(20, truncate=False)
grafico_barh(resultado_p4_1.toPandas(), "linguagem", "total", "Linguagens mais usadas 2025", "Total")

+-----------+-----+
|linguagem  |total|
+-----------+-----+
|Python     |1928 |
|SQL        |1763 |
|R          |327  |
|Scala      |70   |
|C/C++/C#   |49   |
|Rust       |35   |
|Julia      |18   |
|VBA        |1    |
|Java       |0    |
|JavaScript |0    |
|.NET       |0    |
|SAS/Stata  |0    |
|PHP        |0    |
|Matlab     |0    |
|Não utiliza|0    |
+-----------+-----+



**2. Cloud predominante** (mesmo padrão de múltipla escolha, escopo técnico).

In [30]:
clouds = [
    ("AWS", "cloud__amazon_web_services_aws"),
    ("GCP", "cloud__google_cloud_gcp"),
    ("Azure", "cloud__azure_microsoft"),
    ("Oracle Cloud", "cloud__oracle_cloud"),
    ("IBM", "cloud__ibm"),
    ("On Premise/Nenhuma", "cloud__servidores_on_premise_nao_utilizamos_cloud"),
    ("Cloud Própria", "cloud__cloud_propria"),
]
resultado_p4_2 = spark.sql(soma_multipla_escolha(clouds)).withColumnRenamed("categoria", "cloud")
resultado_p4_2.show(truncate=False)
grafico_barh(resultado_p4_2.toPandas(), "cloud", "total", "Cloud mais usada 2025", "Total")

+------------------+-----+
|cloud             |total|
+------------------+-----+
|AWS               |1011 |
|Azure             |730  |
|GCP               |647  |
|On Premise/Nenhuma|300  |
|Cloud Própria     |114  |
|Oracle Cloud      |94   |
|IBM               |25   |
+------------------+-----+



### P5 — Adoção de IA

**1. IA generativa como prioridade na empresa** (categoria, ignoro nulos).

In [31]:
resultado_p5_1 = spark.sql("""
    SELECT ia_prioridade, COUNT(*) AS total,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS percentual
    FROM state_of_data WHERE ia_prioridade IS NOT NULL
    GROUP BY ia_prioridade ORDER BY total DESC
""")
resultado_p5_1.show(truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----+----------+
|ia_prioridade                                                                                                                                          |total|percentual|
+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----+----------+
|Sim, está entre nossas principais prioridades para os próximos 2-4 anos (com discussões de iniciativas e orçamentos de curto a médio prazo).           |240  |36.8      |
|Mais ou menos... É uma das várias iniciativas que estamos impulsionando, mas não é uma prioridade (tratam-se de iniciativas isoladas e com pouco foco).|169  |25.9      |
|Sim, é nossa principal prioridade como empresa (com foco executivo significativo e alocação de orçamento relevante).                            

**2. Quem paga pela IA usada no trabalho** (bloco de uso pessoal, escopo técnico).

In [32]:
uso_ia = [
    ("Não usa", "ia_uso_pessoal__nao_uso_solucoes_de_ai_generativa_com_foco_em_produtividade"),
    ("Usa grátis", "ia_uso_pessoal__uso_solucoes_gratuitas_de_ai_generativa_com_foco_em_produtividade"),
    ("Usa e paga", "ia_uso_pessoal__uso_e_pago_pelas_solucoes_de_ai_generativa_com_foco_em_produtividade"),
    ("Empresa paga", "ia_uso_pessoal__a_empresa_que_trabalho_paga_pelas_solucoes_de_ai_generativa_com_foco_em_produtividade"),
    ("Usa Copilot", "ia_uso_pessoal__uso_solucoes_do_tipo_copilot"),
]
resultado_p5_2 = spark.sql(soma_multipla_escolha(uso_ia)).withColumnRenamed("categoria", "uso_pessoal")
resultado_p5_2.show(truncate=False)
grafico_barh(resultado_p5_2.toPandas(), "uso_pessoal", "total", "Uso pessoal de IA generativa 2025", "Total")

+------------+-----+
|uso_pessoal |total|
+------------+-----+
|Empresa paga|891  |
|Usa grátis  |643  |
|Usa Copilot |628  |
|Usa e paga  |567  |
|Não usa     |44   |
+------------+-----+



### P6 — Diferenças por região, senioridade e modelo de trabalho

**1. Salário por região e senioridade.** Abro por senioridade de propósito (viés de
composição), e restrinjo aos três níveis com ordem lógica.

In [33]:
resultado_p6_1 = spark.sql("""
    SELECT regiao_atual, nivel_senioridade,
           ROUND(AVG(faixa_salarial_num), 0) AS salario_medio_estimado, COUNT(*) AS total
    FROM state_of_data_num
    WHERE aplica_analise_emprego = true AND aplica_analise_tecnica = true
      AND regiao_atual IS NOT NULL AND nivel_senioridade IN ('Júnior', 'Pleno', 'Sênior')
    GROUP BY regiao_atual, nivel_senioridade ORDER BY regiao_atual, nivel_senioridade
""")
resultado_p6_1.show(30, truncate=False)

+------------+-----------------+----------------------+-----+
|regiao_atual|nivel_senioridade|salario_medio_estimado|total|
+------------+-----------------+----------------------+-----+
|Centro-oeste|Júnior           |4113.0                |40   |
|Centro-oeste|Pleno            |8208.0                |53   |
|Centro-oeste|Sênior           |14867.0               |60   |
|Nordeste    |Júnior           |3714.0                |69   |
|Nordeste    |Pleno            |7596.0                |94   |
|Nordeste    |Sênior           |14041.0               |86   |
|Norte       |Júnior           |3929.0                |7    |
|Norte       |Pleno            |7100.0                |10   |
|Norte       |Sênior           |19500.0               |12   |
|Sudeste     |Júnior           |4306.0                |312  |
|Sudeste     |Pleno            |8006.0                |463  |
|Sudeste     |Sênior           |13822.0               |526  |
|Sul         |Júnior           |3988.0                |82   |
|Sul    

In [34]:
import seaborn as sns
pdf = resultado_p6_1.toPandas()
paleta = {"Sênior": "#08306B", "Pleno": "#2171B5", "Júnior": "#6BAED6"}
plt.figure(figsize=(11, 6))
sns.barplot(data=pdf, y="regiao_atual", x="salario_medio_estimado", hue="nivel_senioridade",
            hue_order=list(reversed(ordem_senioridade)), palette=paleta)
plt.title("Salário médio por região e senioridade 2025", fontsize=12, fontweight="bold", pad=15)
plt.xlabel("Salário médio estimado (R$)"); plt.ylabel("Região")
plt.grid(axis="x", linestyle="--", alpha=0.4)
plt.legend(title="Senioridade", bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
sns.despine(); plt.tight_layout(); plt.show()

**2. Salário por modelo de trabalho (aberto por senioridade).**

In [35]:
resultado_p6_2 = spark.sql("""
    SELECT nivel_senioridade, modelo_trabalho_atual,
           ROUND(AVG(faixa_salarial_num), 0) AS salario_medio_estimado, COUNT(*) AS total
    FROM state_of_data_num
    WHERE aplica_analise_emprego = true AND aplica_analise_tecnica = true
    GROUP BY nivel_senioridade, modelo_trabalho_atual
    ORDER BY nivel_senioridade, salario_medio_estimado DESC
""")
resultado_p6_2.show(30, truncate=False)

+-------------------+--------------------------------------------------------------------------------------------------------------+----------------------+-----+
|nivel_senioridade  |modelo_trabalho_atual                                                                                         |salario_medio_estimado|total|
+-------------------+--------------------------------------------------------------------------------------------------------------+----------------------+-----+
|Especialista/Staff+|Modelo 100% remoto                                                                                            |21928.0               |180  |
|Especialista/Staff+|Modelo híbrido flexível (o funcionário tem liberdade para escolher quando estar no escritório presencialmente)|20719.0               |73   |
|Especialista/Staff+|Modelo híbrido com dias fixos de trabalho presencial                                                          |17359.0               |64   |
|Especialista/Staff+|Modelo 

**3. Salário por nível de ensino** (base empregada).

In [36]:
resultado_p6_3 = spark.sql("""
    SELECT nivel_ensino, ROUND(AVG(faixa_salarial_num), 0) AS salario_medio_estimado, COUNT(*) AS total
    FROM state_of_data_num WHERE aplica_analise_emprego = true
    GROUP BY nivel_ensino ORDER BY salario_medio_estimado DESC
""")
resultado_p6_3.show(truncate=False)
grafico_barh(resultado_p6_3.toPandas(), "nivel_ensino", "salario_medio_estimado",
             "Salário médio por nível de ensino 2025", "Salário médio estimado (R$)")

+--------------------------+----------------------+-----+
|nivel_ensino              |salario_medio_estimado|total|
+--------------------------+----------------------+-----+
|Prefiro não informar      |21389.0               |9    |
|Mestrado                  |18212.0               |416  |
|Doutorado ou Phd          |17765.0               |132  |
|Pós-graduação             |14385.0               |1315 |
|Não tenho graduação formal|13197.0               |57   |
|Graduação/Bacharelado     |11475.0               |1003 |
|Estudante de Graduação    |4723.0                |295  |
+--------------------------+----------------------+-----+



**4. Atitude diante de um retorno presencial obrigatório** (base empregada).

In [37]:
import textwrap
resultado_p6_4 = spark.sql("""
    SELECT atitude_retorno_presencial, COUNT(*) AS total
    FROM state_of_data
    WHERE aplica_analise_emprego = true AND atitude_retorno_presencial IS NOT NULL
    GROUP BY atitude_retorno_presencial ORDER BY total DESC
""")
resultado_p6_4.show(truncate=False)
pdf = resultado_p6_4.toPandas()
pdf["fmt"] = pdf["atitude_retorno_presencial"].apply(lambda x: textwrap.fill(str(x), 25))
grafico_barh(pdf, "fmt", "total", "Atitude sobre retorno presencial 2025", "Quantidade de pessoas")

+-----------------------------------------------------------+-----+
|atitude_retorno_presencial                                 |total|
+-----------------------------------------------------------+-----+
|Vou procurar outra oportunidade no modelo híbrido ou remoto|1399 |
|Vou aceitar e retornar ao modelo 100% presencial           |917  |
|Vou procurar outra oportunidade no modelo 100% remoto      |911  |
+-----------------------------------------------------------+-----+



### P7 — Oportunidades e desafios

**1. Satisfação geral** (base empregada; `satisfeito_empresa` já é boolean).

In [38]:
resultado_p7_1 = spark.sql("""
    SELECT CASE WHEN satisfeito_empresa = true THEN 'Satisfeito'
                WHEN satisfeito_empresa = false THEN 'Insatisfeito' END AS satisfeito_empresa,
           COUNT(*) AS total,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS percentual
    FROM state_of_data
    WHERE aplica_analise_emprego = true AND satisfeito_empresa IS NOT NULL
    GROUP BY 1 ORDER BY total DESC
""")
resultado_p7_1.show(truncate=False)
grafico_barh(resultado_p7_1.toPandas(), "satisfeito_empresa", "total",
             "Satisfação com a empresa 2025", "Quantidade de pessoas")

+------------------+-----+----------+
|satisfeito_empresa|total|percentual|
+------------------+-----+----------+
|Satisfeito        |2226 |69.0      |
|Insatisfeito      |1001 |31.0      |
+------------------+-----+----------+



**2. Pretensão de trocar de emprego em 6 meses** (base empregada).

In [39]:
import textwrap
resultado_p7_2 = spark.sql("""
    SELECT pretende_mudar_emprego, COUNT(*) AS total,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS percentual
    FROM state_of_data
    WHERE aplica_analise_emprego = true AND pretende_mudar_emprego IS NOT NULL
    GROUP BY pretende_mudar_emprego ORDER BY total DESC
""")
resultado_p7_2.show(truncate=False)
pdf = resultado_p7_2.toPandas()
pdf["fmt"] = pdf["pretende_mudar_emprego"].apply(lambda x: textwrap.fill(str(x), 30))
grafico_barh(pdf, "fmt", "total", "Pretensão de troca de emprego 2025", "Quantidade de pessoas")

+-----------------------------------------------------------------------+-----+----------+
|pretende_mudar_emprego                                                 |total|percentual|
+-----------------------------------------------------------------------+-----+----------+
|Não estou buscando, mas me considero aberto a outras oportunidades     |1306 |40.5      |
|Estou em busca de oportunidades dentro ou fora do Brasil               |818  |25.3      |
|Não estou buscando e não pretendo mudar de emprego nos próximos 6 meses|804  |24.9      |
|Estou em busca de oportunidades, mas apenas fora do Brasil             |299  |9.3       |
+-----------------------------------------------------------------------+-----+----------+



**3. Principais desafios dos gestores.** Bloco de múltipla escolha com escopo de
gestor — somo cada desafio a partir da lista de colunas.

In [42]:
desafios = [
    ("Contratar talentos", "desafios_gestor__a_contratar_novos_talentos"),
    ("Reter talentos", "desafios_gestor__b_reter_talentos"),
    ("Convencer a empresa a investir", "desafios_gestor__c_convencer_a_empresa_a_aumentar_os_investimentos_na_area_de_dados"),
    ("Gestão de equipes remotas", "desafios_gestor__d_gestao_de_equipes_no_ambiente_remoto"),
    ("Projetos multidisciplinares", "desafios_gestor__e_gestao_de_projetos_envolvendo_areas_multidisciplinares_da_empresa"),
    ("Qualidade/confiabilidade", "desafios_gestor__f_organizar_as_informacoes_e_garantir_a_qualidade_e_confiabilidade"),
    ("Processar/armazenar alto volume", "desafios_gestor__g_conseguir_processar_e_armazenar_um_alto_volume_de_dados"),
    ("Gerar valor para o negócio", "desafios_gestor__h_conseguir_gerar_valor_para_as_areas_de_negocios_atraves_de_estudos_e_experimentos"),
    ("Modelos de ML em produção", "desafios_gestor__i_desenvolver_e_manter_modelos_machine_learning_em_producao"),
    ("Gerenciar expectativa das áreas", "desafios_gestor__j_gerenciar_a_expectativa_das_areas_de_negocio_em_relacao_as_entregas_das_equipes_de_dados"),
    ("Manutenção de projetos/modelos", "desafios_gestor__k_garantir_a_manutencao_dos_projetos_e_modelos_em_producao_em_meio_ao_crescimento_da_empresa"),
    ("Levar inovação", "desafios_gestor__conseguir_levar_inovacao_para_a_empresa_atraves_dos_dados"),
    ("Garantir ROI", "desafios_gestor__garantir_retorno_do_investimento_roi_em_projetos_de_dados"),
    ("Dividir tempo técnico/gestão", "desafios_gestor__dividir_o_tempo_entre_entregas_tecnicas_e_gestao"),
]
resultado_p7_3 = spark.sql(soma_multipla_escolha(desafios, escopo="aplica_analise_gestor")).withColumnRenamed("categoria", "desafio")
resultado_p7_3.show(truncate=False)
grafico_barh(resultado_p7_3.toPandas(), "desafio", "total",
             "Principais desafios dos gestores 2025", "Quantidade de pessoas")

+-------------------------------+-----+
|desafio                        |total|
+-------------------------------+-----+
|Dividir tempo técnico/gestão   |237  |
|Contratar talentos             |0    |
|Reter talentos                 |0    |
|Convencer a empresa a investir |0    |
|Gestão de equipes remotas      |0    |
|Projetos multidisciplinares    |0    |
|Qualidade/confiabilidade       |0    |
|Processar/armazenar alto volume|0    |
|Gerar valor para o negócio     |0    |
|Modelos de ML em produção      |0    |
|Gerenciar expectativa das áreas|0    |
|Manutenção de projetos/modelos |0    |
|Levar inovação                 |0    |
|Garantir ROI                   |0    |
+-------------------------------+-----+



## 4. Exportando as tabelas Gold

Persisto **uma tabela por sub-pergunta** (cada uma tem um grão/`GROUP BY` próprio),
organizadas em 7 pastas por pergunta do desafio. Cada tabela leva a coluna
`ano_pesquisa` e é gravada particionada — com `partitionOverwriteMode = dynamic`,
escrevo só a partição 2025 sem apagar as outras edições no mesmo caminho.

In [43]:
ANO_PESQUISA = 2025

tabelas_por_pergunta = {
    "p1_estrutura_mercado": {
        "situacao_trabalho": resultado_p1_1, "setor_top10": resultado_p1_2,
        "distribuicao_cargos": resultado_p1_3, "distribuicao_senioridade": resultado_p1_4,
        "percentual_gestores": resultado_p1_5, "experiencia_x_senioridade": resultado_p1_6,
        "modelo_trabalho_atual": resultado_p1_7, "modelo_trabalho_ideal": resultado_p1_7_1,
        "porte_empresa": resultado_p1_8,
    },
    "p2_perfis_valorizados": {
        "salario_por_cargo_senioridade": resultado_p2_1, "salario_por_experiencia_dados": resultado_p2_2,
        "salario_por_experiencia_ti": resultado_p2_2_1, "salario_migracao_ti": resultado_p2_3,
        "salario_por_funcao_senioridade": resultado_p2_4, "objetivos_carreira_top5": resultado_p2_5,
    },
    "p3_diversidade": {
        "proporcao_genero": resultado_p3_1, "gap_salarial_genero": resultado_p3_2,
        "raca_por_senioridade": resultado_p3_3,
    },
    "p4_tecnologias": {
        "linguagens_mais_usadas": resultado_p4_1, "cloud_predominante": resultado_p4_2,
    },
    "p5_ia_generativa": {
        "prioridade_ia_empresa": resultado_p5_1, "quem_paga_ia": resultado_p5_2,
    },
    "p6_diferencas_regionais": {
        "salario_por_regiao_senioridade": resultado_p6_1, "salario_por_modelo_trabalho": resultado_p6_2,
        "salario_por_nivel_ensino": resultado_p6_3, "atitude_retorno_presencial": resultado_p6_4,
    },
    "p7_oportunidades_desafios": {
        "satisfacao_geral": resultado_p7_1, "intencao_troca_emprego": resultado_p7_2,
        "desafios_gestores": resultado_p7_3,
    },
}

spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

for pasta, tabelas in tabelas_por_pergunta.items():
    for nome_tabela, df_resultado in tabelas.items():
        caminho = CAMINHO_GOLD_BASE / pasta / nome_tabela
        (df_resultado.withColumn("ano_pesquisa", F.lit(ANO_PESQUISA))
         .write.mode("overwrite").partitionBy("ano_pesquisa").parquet(str(caminho)))
    print(f"{pasta}: {len(tabelas)} tabela(s) exportada(s)")

total_tabelas = sum(len(t) for t in tabelas_por_pergunta.values())
print(f"\nTotal: {total_tabelas} tabelas Gold exportadas. Partição: ano_pesquisa={ANO_PESQUISA}")

p1_estrutura_mercado: 9 tabela(s) exportada(s)
p2_perfis_valorizados: 6 tabela(s) exportada(s)
p3_diversidade: 3 tabela(s) exportada(s)
p4_tecnologias: 2 tabela(s) exportada(s)
p5_ia_generativa: 2 tabela(s) exportada(s)
p6_diferencas_regionais: 4 tabela(s) exportada(s)
p7_oportunidades_desafios: 3 tabela(s) exportada(s)

Total: 29 tabelas Gold exportadas. Partição: ano_pesquisa=2025


## 5. Conclusão

A Gold 2025 ficou com 29 tabelas nas 7 pastas por pergunta, cada uma no grão certo
pra virar gráfico/card na apresentação executiva, e particionada por `ano_pesquisa`
pra empilhar com as demais edições no Athena.